# Authentication

In [ ]:
import getpass
import os

# Read the API key from the environment or request it without echoing.
opeiai_token = os.getenv("OPENAI_API_KEY") or getpass.getpass("OpenAI API key: ")


# Dependencies

In [ ]:
!pip install langgraph
!pip install openai
!pip install sacrebleu
!pip install codebleu
!pip install tree-sitter-python
!pip install langchain_text_splitters

!pip install git+https://github.com/k4black/codebleu.git
!pip install tree-sitter --upgrade


# Imports

In [ ]:
import torch

torch.cuda.is_available()
from accelerate import Accelerator
import numpy as np

# from transformers import AutoTokenizer, AutoModelForCausalLM
# import shap
import matplotlib.pyplot as plt
from IPython.core.display import HTML
import pandas as pd

torch.cuda.empty_cache()
accelerator = Accelerator()
accelerator.free_memory()


import torch
import torch.nn.functional as F


import os
import json


# Dataset

In [ ]:
# human eval
# from datasets import load_dataset
# from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import pandas as pd

# Load the HumanEval dataset using parquetA
df = pd.read_parquet(
    "hf://datasets/openai/openai_humaneval/openai_humaneval/test-00000-of-00001.parquet"
)

# Get the first row
first_row = df.iloc[0]


# Load the model

In [ ]:
import getpass
import os
from openai import OpenAI
import os

os.environ["OPENAI_API_KEY"] = opeiai_token


In [ ]:
# 1. First install the package (do this once):

from openai import OpenAI
import os

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))  # ← never hard-code this!


MODEL = "gpt-3.5-turbo-1106"


In [ ]:
import os
import time


def generator_nonlocal(text: str):
    start_time = time.time()
    response = client.chat.completions.create(
        model=MODEL,  # or "gpt-4o", "o1-mini", "gpt-4.1", etc.
        messages=[
            {
                "role": "system",
                "content": "You are a code generation model tasked with generating code for this instruction.You must provide a complete and correct logic to solve the problem.Do not return a placeholder like `pass` or `#todo`.The implementation must be functional.When the function complete stop .do not generate extra space or newline .generate the full function and then stop.",
            },
            {"role": "user", "content": f"{text}"},
        ],
        temperature=0.0,
        seed=42,
    )
    end_time = time.time()
    duration = end_time - start_time

    return response.choices[0].message.content, response.usage.total_tokens, duration


# Load the embedding model

In [ ]:
from openai import OpenAI

client_embedding = OpenAI(api_key=opeiai_token)

# text = "def add(a, b): return a + b"

# response = client.embeddings.create(
#     model="text-embedding-3-large",
#     input=text
# )

# embedding = response.data[0].embedding

# print(len(embedding))   # Vector length
# print(embedding[:10])   # First few values


# Modules

## Code extraction

In [ ]:
import re


def extract_python_code(markdown_text: str):
    """
    Extracts and returns the first Python code block fenced with ```python
    If no such block exists, returns the original text.
    """
    pattern = r"```python\s*(.*?)```"
    match = re.search(pattern, markdown_text, re.DOTALL | re.IGNORECASE)

    if match:
        return match.group(1).strip()  # Return the code inside the fences
    else:
        return markdown_text


# Example usage:
text_with_code = """
Here's some explanation.

```python
def hello():
    print("Hello, World!")
    ```
    kjhkjh
    """

extracted_code = extract_python_code(text_with_code)
print(extracted_code)


## Text chunking

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language

text = """def closest_integer(value):
    ''' Create a function that takes a value (string) representing a number and returns the closest integer to it. If the number is equidistant from two integers, round it away from zero.
    Examples
    >>> closest_integer("10")
    10
    >>> closest_integer("15.3")
    15
    Note: Rounding away from zero means that if the given number is equidistant from two integers, the one you should return is the one that is the farthest from zero. For example closest_integer("14.5") should return 15 and closest_integer("-14.5") should return -15.
    '''
    pass  # your implementation here
"""
text1 = "Write a function to find the longest chain which can be formed from the given set of pairs."


def get_semantic_chunks(text: str) -> list[str]:

    splitter = RecursiveCharacterTextSplitter(
        # language=Language.PYTHON,
        chunk_size=200,  # ← tune this: 400–800 range often ideal
        chunk_overlap=0,  # keeps some context across boundaries
        separators=["\n\n", "\n", ".", ","],
    )

    # # Better alternative for bigger codebases
    # from llama_index.core.node_parser import CodeSplitter

    # splitter = CodeSplitter(
    #     language="python",
    #     chunk_lines=50,          # ~ lines per chunk
    #     chunk_lines_overlap=10,
    # )

    chunks = splitter.split_text(text)
    return chunks


chunks = get_semantic_chunks(text)
print(len(chunks))


In [ ]:
def split_into_three_parts(text):
    """
    Splits a string into 3 parts of as equal length as possible.
    Returns a list with 3 strings.
    """
    if not text or text.strip() == "":
        return ["", "", ""]

    words = text.split()
    n = len(words)

    part_size = n // 3
    remainder = n % 3

    # Calculate sizes for each chunk
    size1 = part_size + (1 if remainder > 0 else 0)
    size2 = part_size + (1 if remainder > 1 else 0)
    size3 = part_size  # remainder == 2 or 1 handled above

    part1 = words[:size1]
    part2 = words[size1 : size1 + size2]
    part3 = words[size1 + size2 :]

    return [" ".join(part1), " ".join(part2), " ".join(part3)]


# Example usage:
text = "This is a sample text that we want to split into three roughly equal parts."
parts = split_into_three_parts(text)
print(parts)


In [ ]:
def process_dict(items):
    n = len(items)
    if n < 3:
        return False
    if n == 3:
        return [str(x) for x in items]

    base = n // 3
    extra = n % 3

    result = []
    start = 0

    for i in range(3):
        size = base + (1 if i < extra else 0)
        chunk = [str(x) for x in items[start : start + size]]
        result.append("\n".join(chunk))
        start += size

    return result


di = ["hello", "hi", "boo", "app"]
r = process_dict(di)


## Metrics

In [ ]:
from sacrebleu.metrics import CHRF
import numpy as np
from typing import List, Dict

# ────────────────────────────────────────────────
#   Initialize ChrF++ (word_order=2 = ChrF++)
# ────────────────────────────────────────────────

chrf = CHRF(
    char_order=6,  # max char n-gram order (default 6)
    word_order=2,  # word n-gram order (2 = ChrF++)
    beta=2,  # F-score beta (default 2)
    lowercase=False,  # keep case-sensitive (recommended for code)
    whitespace=True,  # consider whitespace (important for code indentation)
    # num_refs=1           # single reference
)

# ────────────────────────────────────────────────
#   Similarity function (0–1 score)
# ────────────────────────────────────────────────


def chrf_plus_plus_similarity(code_a: str, code_b: str) -> float:
    """
    Returns ChrF++ similarity score (0–1).
    Higher = more similar.
    """
    score = chrf.corpus_score(
        hypotheses=[code_a], references=[[code_b]]  # references is list of lists
    )
    return score.score / 100.0  # sacrebleu returns 0–100, we normalize to 0–1


In [ ]:
def sensivity(phi1, phi2):
    """
    Official SENS_MAX with L2 norm (as in the original math formula)
    Works for any lengths by smart resizing + zero-padding fallback
    """
    a = np.array(phi1, dtype=float)
    b = np.array(phi2, dtype=float)

    # Make both the same length (longest one wins)
    new_len = max(len(a), len(b))
    x = np.linspace(0, 1, new_len)

    # Interpolate both to the new grid (this is what papers do)
    a_resized = np.interp(x, np.linspace(0, 1, len(a)), a, left=0, right=0)
    b_resized = np.interp(x, np.linspace(0, 1, len(b)), b, left=0, right=0)

    # This is the exact ∥Φ(y) − Φ(x)∥ from the formula
    return np.linalg.norm(a_resized - b_resized)  # ← L₂ norm (official!)


In [ ]:
# Make sure you have installed the package first:

from codebleu import calc_codebleu  # ← this is the correct import


def calculate_codebleu(
    prediction: str,
    reference: str,
) -> dict:
    """
    Compute CodeBLEU score between one predicted code string and one reference code string.

    Returns a dict like:
    {
        'codebleu': final score,
        'ngram_match_score': ...,
        'weighted_ngram_match_score': ...,
        'syntax_match_score': ...,
        'dataflow_match_score': ...
    }
    """
    try:
        lang = "python"
        weights = (
            0.1,
            0.1,
            0.4,
            0.4,
        )  # n-gram, weighted n-gram, syntax (AST), dataflow
        # The function expects *lists* even for single examples

        result = calc_codebleu(
            [reference],  # list of reference(s)
            [prediction],  # list of prediction(s)
            lang=lang,
            weights=weights,
        )

        return result
    except Exception as e:
        return {"codebleu": False}


# ────────────────────────────────────────────────
# Example usage (copy-paste ready)
# ────────────────────────────────────────────────

if __name__ == "__main__":
    ref = """
def add(a, b):
    return a + b
"""

    gen = """
def add(x, y):
    result = x + y
    return result
"""

    score = calculate_codebleu(gen, ref)

    print("CodeBLEU result:")
    print(score)

    # Most people just want the main score:
    print(f"\nFinal CodeBLEU score: {score['codebleu']:.4f}")


## Feature ablation and perturbation

In [ ]:
def ablations_features(items):
    """
    Generate all possible feature combinations by removing exactly one item each time.
    Returns list of space-joined strings.

    Examples:
      ["red", "big", "car"]   →  ["big car", "red car", "red big"]
      ["A", "B"]              →  ["B", "A"]
      ["X"]                   →  [""]
      []                      →  []
      ["a","b","c","d"]      →  ["b c d", "a c d", "a b d", "a b c"]
    """
    if not items:
        return []

    result = []
    n = len(items)

    for i in range(n):
        # take all elements except the i-th one
        ablated = items[:i] + items[i + 1 :]
        # join with space (empty list → empty string)
        result.append(" ".join(ablated))

    return result


# ────────────────────────────────────────────────
# Usage examples
# ────────────────────────────────────────────────

features = ["speed", "power", "control"]

ablations = ablations_features(features)


In [ ]:
import random
from typing import List


def add_small_noise(text, noise_level=0.1, seed=None, rng=None):
    """
    Add small perturbations to text without changing its meaning drastically.
    """
    if not text or len(text.strip()) == 0:
        return text

    words = text.split()
    if len(words) == 0:
        return text

    # Create a random number generator if none was provided.
    if rng is None:
        rng = random.Random(seed)

    n_changes = max(1, int(len(words) * noise_level))

    for _ in range(n_changes):
        if len(words) < 2:
            break

        change_type = rng.choice(["swap", "typo", "duplicate"])
        idx = rng.randint(0, len(words) - 1)

        if change_type == "swap" and len(words) > 1:
            j = (idx + 1) % len(words)
            words[idx], words[j] = words[j], words[idx]
        elif change_type == "typo" and len(words[idx]) > 2:
            word = list(words[idx])
            pos = rng.randint(0, len(word) - 1)
            word[pos] = chr(ord(word[pos]) + rng.choice([-1, 1]))
            words[idx] = "".join(word)
        elif change_type == "duplicate":
            words.insert(idx, words[idx])

    return " ".join(words)


def perturb_input_sensitivity(input_parts: List[str], noise_level: float = 0.1):
    # Share one RNG across all parts so a single seed makes the output reproducible.
    # seed= 42
    # seed= 123
    # seed= 314
    # seed= 456
    # seed= 512
    # seed= 999
    # seed= 1234
    # seed= 7
    # seed= 0
    seed = 999

    rng = random.Random(seed)

    processed_parts = []
    for part in input_parts:
        perturbed_part = add_small_noise(part, noise_level=noise_level, rng=rng)
        processed_parts.append(perturbed_part)

    return processed_parts


input_examples = [
    "def has_close_elements(numbers: List[float], threshold: float) -> bool:",
    "",
]
kernel_shap_values = [5.1414, 11.7640, 7.3399]

# A fixed seed produces the same output on every run.
perturbed_input_list = perturb_input_sensitivity(input_examples, 0.5)
print(perturbed_input_list)


## Embeddings

In [ ]:
def cosine_similarity_explicit(a: np.ndarray, b: np.ndarray) -> float:
    """
    Classic implementation (with safety checks).
    Use this version if you're not 100% sure the vectors are already normalized.
    """
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)

    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)

    if norm_a == 0 or norm_b == 0:
        return 0.0

    return np.dot(a, b) / (norm_a * norm_b)


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

# model.eval()          # usually better for inference
# model.to("cuda")    # uncomment if you have GPU


def get_code_embedding(code: str) -> torch.Tensor:
    response = client_embedding.embeddings.create(
        model="text-embedding-3-large", input=f"{code}"
    )

    # Convert to NumPy for averaging.
    # token_vectors = np.array(response.data[0].embedding)

    # mean pooling → Average the vectors of the actual tokens
    # mean_emb = token_vectors.mean(axis=0)   # Shape: (768,)

    emb = response.data[0].embedding
    return torch.tensor(emb, dtype=torch.float32)

    # return mean_emb
    # print(len(embedding))


# ────────────────────────────────────────────────
code = """
def quicksort(arr):
    if len(arr) <= 1:
        return arr
    pivot = arr[len(arr) // 2]
    left = [x for x in arr if x < pivot]
    middle = [x for x in arr if x == pivot]
    right = [x for x in arr if x > pivot]
    return quicksort(left) + middle + quicksort(right)
"""

emb = get_code_embedding(code)
# print(emb.shape)           # torch.Size([768])
# print(emb[:5])             # first few values


In [ ]:
def cosine_similarity(a: torch.Tensor, b: torch.Tensor) -> float:
    return torch.nn.functional.cosine_similarity(a, b, dim=0).item()


code1 = """
function factorial(n) {
    if (n === 0 || n === 1) return 1;
    return n * factorial(n - 1);
}
"""

code2 = """
def factorial(n):
    if n in (0, 1):
        return 1
    return n * factorial(n - 1)
"""

emb1 = get_code_embedding(code1)
emb2 = get_code_embedding(code2)

similarity = cosine_similarity(emb1, emb2)
print(f"Similarity: {similarity:.4f}")  # usually 0.82–0.93 range


In [ ]:
import numpy as np


def cosine_similarity_L2(vec1, vec2):
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)

    dot = np.dot(vec1, vec2)
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)

    if norm1 == 0 or norm2 == 0:
        return 0

    return dot / (norm1 * norm2)


# Example:
v1 = [1, 2, 3]
v2 = [2, 4, 6]
print(cosine_similarity_L2(v1, v2))


## Spread score

In [ ]:
import math


def standard_deviation(lst):
    n = len(lst)
    mean = sum(lst) / n
    variance = sum((x - mean) ** 2 for x in lst) / n
    std = math.sqrt(variance)
    return std


numbers = [10, 12, 23, 23, 16, 23, 21, 16]
print(standard_deviation(numbers))


# Evaluation workflow

In [ ]:
dataset = df.iloc[89:]


In [ ]:
import csv

for index, row in dataset.iterrows():

    prompt = row["prompt"]
    test = row["test"]

    list_of_chunks = get_semantic_chunks(prompt)
    r = process_dict(list_of_chunks)

    if r is False:
        list_of_chunks = split_into_three_parts(prompt)

    else:
        list_of_chunks = r

    input_examples = list_of_chunks

    input_examples1 = input_examples

    ablated_features = ablations_features(input_examples1)

    code, token_consumption_code, duration = generator_nonlocal(f"{prompt}")
    code = extract_python_code(code)

    ablatedd_code_phi = []
    sim_chrff_phi = []
    sim_embedd_phi = []
    embedd_lower_phi = []
    codebleu_phi = []

    time_codebleu_phi = 0
    time_chrf_phi = 0
    time_nomic_phi = 0
    time_embed_jinja_phi = 0

    start_time_nomic = time.time()
    end_time_nomic = time.time()

    time_nomic = end_time_nomic - start_time_nomic

    start_time_jinja = time.time()
    embeddings_lower2 = get_code_embedding(code)
    end_time_jinja = time.time()

    time_embed_jinja = end_time_jinja - start_time_jinja

    token_consumption_ablation_coder = 0
    duration_ablation_coder = 0

    for i, item in enumerate(ablated_features, 1):

        ablated_code, token_consumption_ablation_coder, duration_ablation_coder = (
            generator_nonlocal(f"{item}")
        )
        token_consumption_ablation_coder += token_consumption_ablation_coder
        duration_ablation_coder += duration_ablation_coder
        ablated_code = extract_python_code(ablated_code)
        ablatedd_code_phi.append(ablated_code)

        start_time_nomic = time.time()
        end_time_nomic = time.time()

        time_nomic_phi = end_time_nomic - start_time_nomic
        time_nomic_phi += time_nomic_phi

        start_time_jinja = time.time()
        embeddings_lower1 = get_code_embedding(ablated_code)
        end_time_jinja = time.time()

        time_embed_jinja_phi = end_time_jinja - start_time_jinja
        time_embed_jinja_phi += time_embed_jinja_phi

        sim = cosine_similarity(embeddings_lower1, embeddings_lower2)
        sim = 1 - sim
        embedd_lower_phi.append(sim)

        start_time_chrf = time.time()
        sim = chrf_plus_plus_similarity(ablated_code, code)
        sim = 1 - sim
        end_time_chrf = time.time()

        time_chrf_phi = start_time_chrf - end_time_chrf
        time_chrf_phi += time_chrf_phi

        sim_chrff_phi.append(sim)

        start_time_codebleu = time.time()
        score = calculate_codebleu(ablated_code, code)
        end_time_codebleu = time.time()
        time_codebleu_phi = start_time_codebleu - end_time_codebleu
        time_codebleu_phi += time_codebleu_phi

        if score["codebleu"] != False:
            score = 1 - score["codebleu"]
            codebleu_phi.append(score)
        else:
            codebleu_phi.append(-1)

    std_chrff = standard_deviation(sim_chrff_phi)
    std_embedd_lower = standard_deviation(embedd_lower_phi)
    std_codebleu = standard_deviation(codebleu_phi)

    perturbed_samples = perturb_input_sensitivity(
        input_parts=input_examples1, noise_level=0.2
    )

    sim_noise = []

    for i in range(3):
        original_prompt = get_code_embedding(input_examples1[i])
        noised_prompt = get_code_embedding(perturbed_samples[i])
        sim_noisee = cosine_similarity_L2(noised_prompt, original_prompt)
        sim_noisee = 1 - sim_noisee

        sim_noise.append(sim_noisee)

    mean_value = max(sim_noise)

    sensitivity_ablated_code = []
    sim_chrff_sens = []
    sim_embedd_sens = []
    embedd_lower_sens = []
    codebleu_sens = []

    time_codebleu_sens = 0
    time_chrf_sens = 0
    time_nomic_sens = 0
    time_embed_jinja_sens = 0

    token_consumption_ablation_coder_sens = 0
    duration_ablation_coder_sens = 0

    for i, item in enumerate(perturbed_samples, 1):

        (
            ablated_code_s,
            token_consumption_ablation_coder_sens,
            duration_ablation_coder_sens,
        ) = generator_nonlocal(f"{item}")
        token_consumption_ablation_coder_sens += token_consumption_ablation_coder_sens
        duration_ablation_coder_sens += duration_ablation_coder_sens
        ablated_code_s = extract_python_code(ablated_code_s)
        sensitivity_ablated_code.append(ablated_code_s)

        start_time_nomic = time.time()
        end_time_nomic = time.time()

        time_nomic_sens = end_time_nomic - start_time_nomic
        time_nomic_sens += time_nomic_sens

        start_time_jinja = time.time()
        embeddings_lower1 = get_code_embedding(ablated_code_s)
        end_time_jinja = time.time()

        time_embed_jinja_sens = end_time_jinja - start_time_jinja
        time_embed_jinja_sens += time_embed_jinja_sens

        sim = cosine_similarity(embeddings_lower1, embeddings_lower2)
        sim = 1 - sim
        embedd_lower_sens.append(sim)

        start_time_chrf = time.time()
        sim = chrf_plus_plus_similarity(ablated_code_s, code)
        sim = 1 - sim
        end_time_chrf = time.time()

        time_chrf_sens = start_time_chrf - end_time_chrf
        time_chrf_sens += time_chrf_sens

        sim_chrff_sens.append(sim)

        start_time_codebleu = time.time()
        score = calculate_codebleu(ablated_code_s, code)
        end_time_codebleu = time.time()
        time_codebleu_sens = start_time_codebleu - end_time_codebleu
        time_codebleu_sens += time_codebleu_sens

        if score["codebleu"] != False:
            sim = 1 - score["codebleu"]
            codebleu_sens.append(sim)
        else:
            codebleu_sens.append(-1)

    sensivity_chrf = sensivity(sim_chrff_sens, sim_chrff_phi)
    sensivity_embedd_lower = sensivity(embedd_lower_sens, embedd_lower_phi)
    sensivity_codebleu = sensivity(codebleu_sens, codebleu_phi)

    new_row = [
        index + 1,
        test,
        list_of_chunks,
        code,
        token_consumption_code,
        duration,
        ablatedd_code_phi,
        sim_chrff_phi,
        sim_embedd_phi,
        embedd_lower_phi,
        codebleu_phi,
        time_codebleu_phi,
        time_chrf_phi,
        time_nomic_phi,
        time_embed_jinja_phi,
        time_nomic,
        time_embed_jinja,
        perturbed_samples,
        sensitivity_ablated_code,
        sim_chrff_sens,
        sim_embedd_sens,
        embedd_lower_sens,
        codebleu_sens,
        time_codebleu_sens,
        time_chrf_sens,
        time_nomic_sens,
        time_embed_jinja_sens,
        sensivity_chrf,
        sensivity_embedd_lower,
        sensivity_codebleu,
        token_consumption_ablation_coder,
        duration_ablation_coder,
        token_consumption_ablation_coder_sens,
        duration_ablation_coder_sens,
        sim_noise,
        mean_value,
        std_chrff,
        std_embedd_lower,
        std_codebleu,
    ]

    with open(
        "/content/outputs_inf_sens_humaneval_gpt3_semantic.csv", "a", newline=""
    ) as file:
        writer = csv.writer(file)
        writer.writerow(new_row)
    print(f"task{index+1}done.")


In [ ]:
# DRIVE_PATH = "/content/drive/MyDrive/outputs_inf_sens_humaneval_gpt3_semantic.csv"
LOCAL_PATH = "/content/outputs_inf_sens_humaneval_gpt3_semantic.csv"


In [ ]:
import pandas as pd
from google.colab import files

files.download(LOCAL_PATH)


# Save in Google Colab

In [ ]:
# file_name='outputs_inf_sens_humaneval_gpt3_nonsemantic.csv'  #change this for every dataset and evaluation


In [ ]:
# # Step 2: Specify the CSV file name/path in the current Colab runtime
# # Change this if your file is in a subfolder, e.g., 'data/my_data.csv'
# source_file = f'/content/{file_name}'          # ←← Change this to your actual file name

# # Step 3: Specify where you want to save it in your Google Drive
# # Example paths:
# # '/content/drive/MyDrive/my_data.csv'                  → root of My Drive
# # '/content/drive/MyDrive/MyFolder/my_data.csv'         → inside a folder
# # '/content/drive/MyDrive/Data/Results/my_data.csv'     → nested folders
# destination_file = f'/content/drive/MyDrive/{file_name}'  # ←← Change this path as needed

# # Step 4: Copy the file from Colab runtime to Drive (overwrites if exists)
# shutil.copy(source_file, destination_file)

# print(f"CSV file '{source_file}' has been saved/overwritten to Google Drive at:")
# print(destination_file)


# Semantic chunking